In [1]:
import pandas as pd

# Load flight data
df = pd.read_csv('/Users/sara.alsiyat/Desktop/Uni/Study/Winter/420 - Database systems/Final Project/Data/01_OnTime_Performance/merged/OnTime_2023_2025_ALL.csv')

print(df.columns)

ParserError: Error tokenizing data. C error: Expected 42 fields in line 7395172, saw 43


In [ ]:
bad_years = df[~df['YEAR'].astype(str).str.isnumeric()]
print(bad_years.head())

In [ ]:
# Count departures by airport and month
departures = df.groupby([
    "YEAR", "QUARTER", "MONTH",
    "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_STATE_NM"
]).size().reset_index(name="Departures_Performed")

# Count arrivals
arrivals = df.groupby([
    "YEAR", "QUARTER", "MONTH",
    "DEST", "DEST_CITY_NAME", "DEST_STATE_NM"
]).size().reset_index(name="Arrivals_Performed")

# Rename DEST fields to match ORIGIN fields for the merge
arrivals = arrivals.rename(columns={
    "DEST": "ORIGIN",
    "DEST_CITY_NAME": "ORIGIN_CITY_NAME",
    "DEST_STATE_NM": "ORIGIN_STATE_NM"
})

# Merge departures + arrivals
traffic = departures.merge(
    arrivals,
    on=["YEAR", "QUARTER", "MONTH", "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_STATE_NM"],
    how="outer"
)

# Fill missing counts with 0 (only the count columns)
traffic[["Departures_Performed", "Arrivals_Performed"]] = traffic[
    ["Departures_Performed", "Arrivals_Performed"]
].fillna(0)

# Calculate totals
traffic["Total_Operations"] = traffic["Departures_Performed"] + traffic["Arrivals_Performed"]

# Final rename for clean output
traffic = traffic.rename(columns={
    "YEAR": "Year",
    "QUARTER": "Quarter",
    "MONTH": "Month",
    "ORIGIN": "Airport_Code",
    "ORIGIN_CITY_NAME": "City",
    "ORIGIN_STATE_NM": "State"
})

# Optional: reorder columns to match your original output order
traffic = traffic[[
    "Year", "Quarter", "Month",
    "Airport_Code", "City", "State",
    "Departures_Performed", "Arrivals_Performed", "Total_Operations"
]]


# Save
traffic.to_csv('/Users/sara.alsiyat/Desktop/Uni/Study/Winter/420 - Database systems/Final Project/Data/04_Airport_Traffic_Data/Airport_Traffic_2023_2025.csv', index=False)

print(f"✅ Created airport traffic data")
print(f"✅ Total airports: {traffic['Airport_Code'].nunique()}")

In [2]:
df = pd.read_csv("/Users/sara.alsiyat/Desktop/Uni/Study/Winter/420 - Database systems/Final Project/Data/01_OnTime_Performance/merged/OnTime_2024_ALL.csv", engine="python", on_bad_lines="skip")

print("Total rows in 2024:", len(df))
print("Unique months:", df['MONTH'].nunique())
print("Unique airports:", df['ORIGIN'].nunique())

Total rows in 2024: 547271
Unique months: 1
Unique airports: 334
